# 05 — Final Comparison and Recommendation

Pull together Scenario 2 and Scenario 5 outputs, compare the best model in
each family, and print the final deployment recommendation. This notebook
reads saved CSVs only; it does not retrain models.

In [1]:
# --- bootstrap: make src/ importable when notebook started outside `uv run` ---
import sys
from pathlib import Path

_HERE = Path.cwd()
for parent in [_HERE, *_HERE.parents]:
    if (parent / "src" / "ujiindoorloc").is_dir():
        if str(parent / "src") not in sys.path:
            sys.path.insert(0, str(parent / "src"))
        break

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [2]:
import pandas as pd
from ujiindoorloc.constants import FIGURES_DIR, MODELS_DIR, TABLES_DIR

## Load Scenario 2 results

In [3]:
s2 = pd.read_csv(TABLES_DIR / "scenario_2_model_metrics.csv")
s2_sorted = s2.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(s2_sorted)

,model,family,accuracy,balanced_accuracy,macro_f1,weighted_f1,fit_seconds,predict_seconds,n_classes
0,random_forest,non_parametric,0.908191,0.892131,0.892380,0.909477,0.426148,0.027256,13
1,logistic_regression,parametric,0.882088,0.884657,0.869955,0.883697,0.224099,0.000902,13
2,lda,parametric,0.888389,0.871823,0.846671,0.890585,0.328724,0.000260,13
3,knn,non_parametric,0.793879,0.802673,0.793154,0.795831,0.027967,0.062718,13
4,qda_pca50,parametric,0.737174,0.767706,0.730904,0.742134,0.069533,0.001519,13
5,decision_tree,non_parametric,0.686769,0.664201,0.676364,0.701898,0.285124,0.000480,13
6,qda_raw,parametric,0.601260,0.615589,0.582956,0.603226,0.383462,0.012762,13


## Load Scenario 5 results

In [4]:
s5_pca = pd.read_csv(TABLES_DIR / "scenario_5_pca_metrics.csv")
s5_lda = pd.read_csv(TABLES_DIR / "scenario_5_lda_projection_metrics.csv")

display(s5_pca.sort_values(["classifier_name", "n_components"]).reset_index(drop=True))
display(s5_lda.sort_values(["classifier_name", "n_components"]).reset_index(drop=True))

,reduction_method,classifier_name,n_components,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,PCA,knn,2,0.306931,0.303705,0.277112,0.315338
1,PCA,knn,5,0.459046,0.506502,0.463576,0.458380
2,PCA,knn,10,0.581458,0.625710,0.600685,0.584271
3,PCA,knn,20,0.790279,0.784296,0.776413,0.791980
4,PCA,knn,30,0.813681,0.798886,0.795786,0.814501
5,PCA,knn,50,0.817282,0.803717,0.800470,0.818101
6,PCA,knn,75,0.815482,0.804331,0.799105,0.816574
7,PCA,knn,100,0.819982,0.811764,0.808263,0.821496
8,PCA,logistic_regression,2,0.252025,0.245923,0.202484,0.230220
9,PCA,logistic_regression,5,0.381638,0.428332,0.363586,0.378684


,reduction_method,classifier_name,n_components,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,LDA_projection,knn,2,0.632763,0.537249,0.515062,0.633525
1,LDA_projection,knn,5,0.826283,0.797956,0.789990,0.827946
2,LDA_projection,knn,10,0.889289,0.870163,0.855930,0.890139
3,LDA_projection,knn,12,0.893789,0.884601,0.865674,0.893810
4,LDA_projection,logistic_regression,2,0.530153,0.412149,0.372626,0.519859
5,LDA_projection,logistic_regression,5,0.841584,0.806072,0.794549,0.842969
6,LDA_projection,logistic_regression,10,0.905491,0.889765,0.881538,0.905063
7,LDA_projection,logistic_regression,12,0.892889,0.884956,0.867723,0.892555


## Best model per family

In [5]:
def best_by_balanced_accuracy(df: pd.DataFrame) -> pd.Series:
    return df.sort_values(
        ["balanced_accuracy", "macro_f1"], ascending=[False, False]
    ).iloc[0]


best_parametric_original = best_by_balanced_accuracy(s2[s2["family"] == "parametric"])
best_non_parametric_original = best_by_balanced_accuracy(s2[s2["family"] == "non_parametric"])
best_pca_reduced = best_by_balanced_accuracy(s5_pca)
best_lda_reduced = best_by_balanced_accuracy(s5_lda)
best_s2 = best_by_balanced_accuracy(s2)
best_s5 = best_by_balanced_accuracy(pd.concat([s5_pca, s5_lda], ignore_index=True))

lr_original = s2[s2["model"] == "logistic_regression"].iloc[0]
rf_rows = s2[s2["model"] == "random_forest"]
rf_original = rf_rows.iloc[0] if not rf_rows.empty else None

if lr_original["balanced_accuracy"] >= best_s2["balanced_accuracy"] - 0.03:
    final_name = "logistic_regression"
    final_source = "Scenario 2 original WAP features"
    final_reason = "within 0.03 balanced_accuracy of the best Scenario 2 model and most explainable"
    final_metrics = lr_original
elif rf_original is not None and rf_original["balanced_accuracy"] > lr_original["balanced_accuracy"] + 0.03:
    final_name = "random_forest"
    final_source = "Scenario 2 original WAP features"
    final_reason = "Random Forest is more than 0.03 balanced_accuracy better than Logistic Regression"
    final_metrics = rf_original
else:
    final_name = best_s2["model"]
    final_source = "Scenario 2 original WAP features"
    final_reason = "best overall Scenario 2 balanced_accuracy after tie-breaking"
    final_metrics = best_s2

family_rows = [
    {
        "family": "best_parametric_original",
        "source": "Scenario 2",
        "model": best_parametric_original["model"],
        "reduction_method": "none",
        "n_components": pd.NA,
        "balanced_accuracy": best_parametric_original["balanced_accuracy"],
        "macro_f1": best_parametric_original["macro_f1"],
    },
    {
        "family": "best_non_parametric_original",
        "source": "Scenario 2",
        "model": best_non_parametric_original["model"],
        "reduction_method": "none",
        "n_components": pd.NA,
        "balanced_accuracy": best_non_parametric_original["balanced_accuracy"],
        "macro_f1": best_non_parametric_original["macro_f1"],
    },
    {
        "family": "best_pca_reduced",
        "source": "Scenario 5",
        "model": best_pca_reduced["classifier_name"],
        "reduction_method": "PCA",
        "n_components": int(best_pca_reduced["n_components"]),
        "balanced_accuracy": best_pca_reduced["balanced_accuracy"],
        "macro_f1": best_pca_reduced["macro_f1"],
    },
    {
        "family": "best_lda_reduced",
        "source": "Scenario 5",
        "model": best_lda_reduced["classifier_name"],
        "reduction_method": "LDA_projection",
        "n_components": int(best_lda_reduced["n_components"]),
        "balanced_accuracy": best_lda_reduced["balanced_accuracy"],
        "macro_f1": best_lda_reduced["macro_f1"],
    },
    {
        "family": "final_recommended_model",
        "source": final_source,
        "model": final_name,
        "reduction_method": "none",
        "n_components": pd.NA,
        "balanced_accuracy": final_metrics["balanced_accuracy"],
        "macro_f1": final_metrics["macro_f1"],
        "reason": final_reason,
    },
]

best_family_df = pd.DataFrame(family_rows)
best_family_df.to_csv(TABLES_DIR / "final_best_model_per_family.csv", index=False)
display(best_family_df)

,family,source,model,reduction_method,n_components,balanced_accuracy,macro_f1,reason
0,best_parametric_original,Scenario 2,logistic_regression,none,<NA>,0.884657,0.869955,NaN
1,best_non_parametric_original,Scenario 2,random_forest,none,<NA>,0.892131,0.892380,NaN
2,best_pca_reduced,Scenario 5,logistic_regression,PCA,100,0.877297,0.866754,NaN
3,best_lda_reduced,Scenario 5,logistic_regression,LDA_projection,10,0.889765,0.881538,NaN
4,final_recommended_model,Scenario 2 original WAP features,logistic_regression,none,<NA>,0.884657,0.869955,within 0.03 balanced_accuracy of the best Scen...


## Final compact comparison

In [6]:
n_waps = "~465 WAPs"
compact_rows = [
    {
        "approach": "Original WAP features",
        "input_features": n_waps,
        "model": best_parametric_original["model"],
        "n_features_or_components": n_waps,
        "balanced_accuracy": best_parametric_original["balanced_accuracy"],
        "macro_f1": best_parametric_original["macro_f1"],
        "explainability": "high",
        "recommendation": "safe baseline",
    },
    {
        "approach": "Original WAP features",
        "input_features": n_waps,
        "model": best_non_parametric_original["model"],
        "n_features_or_components": n_waps,
        "balanced_accuracy": best_non_parametric_original["balanced_accuracy"],
        "macro_f1": best_non_parametric_original["macro_f1"],
        "explainability": "medium",
        "recommendation": "best performance candidate",
    },
    {
        "approach": "PCA components",
        "input_features": "compressed PCs",
        "model": best_pca_reduced["classifier_name"],
        "n_features_or_components": f"{int(best_pca_reduced['n_components'])} PCs",
        "balanced_accuracy": best_pca_reduced["balanced_accuracy"],
        "macro_f1": best_pca_reduced["macro_f1"],
        "explainability": "lower",
        "recommendation": "good compression",
    },
    {
        "approach": "LDA projection",
        "input_features": "supervised LDs",
        "model": best_lda_reduced["classifier_name"],
        "n_features_or_components": f"{int(best_lda_reduced['n_components'])} LDs",
        "balanced_accuracy": best_lda_reduced["balanced_accuracy"],
        "macro_f1": best_lda_reduced["macro_f1"],
        "explainability": "medium",
        "recommendation": "supervised DR",
    },
]

compact_df = pd.DataFrame(compact_rows)
compact_df.to_csv(TABLES_DIR / "final_compact_model_comparison.csv", index=False)
display(compact_df)

,approach,input_features,model,n_features_or_components,balanced_accuracy,macro_f1,explainability,recommendation
0,Original WAP features,~465 WAPs,logistic_regression,~465 WAPs,0.884657,0.869955,high,safe baseline
1,Original WAP features,~465 WAPs,random_forest,~465 WAPs,0.892131,0.892380,medium,best performance candidate
2,PCA components,compressed PCs,logistic_regression,100 PCs,0.877297,0.866754,lower,good compression
3,LDA projection,supervised LDs,logistic_regression,10 LDs,0.889765,0.881538,medium,supervised DR


## Final recommendation logic

In [7]:
notes = []
if lr_original["balanced_accuracy"] >= best_s2["balanced_accuracy"] - 0.03:
    notes.append(
        "Logistic Regression is within 0.03 balanced_accuracy of the best "
        "original-feature model, so it is the safest explainable choice."
    )

if rf_original is not None and rf_original["balanced_accuracy"] > lr_original["balanced_accuracy"] + 0.03:
    notes.append(
        "Random Forest is more than 0.03 balanced_accuracy better than "
        "Logistic Regression, so use it as the practical best model and "
        "keep Logistic Regression as the explainable baseline."
    )

if best_pca_reduced["balanced_accuracy"] >= best_s2["balanced_accuracy"] - 0.02:
    notes.append(
        f"PCA with {int(best_pca_reduced['n_components'])} components is "
        "within 0.02 balanced_accuracy of the original-feature best, so it "
        "is a valid compression strategy."
    )

if best_lda_reduced["balanced_accuracy"] >= best_s2["balanced_accuracy"] - 0.03:
    notes.append(
        f"LDA projection with {int(best_lda_reduced['n_components'])} "
        "components performs close to the original-feature best, making it "
        "a strong supervised dimensionality-reduction option."
    )

print("Final recommendation:")
print(
    f"Recommend {final_name} from {final_source}. It reaches "
    f"balanced_accuracy={final_metrics['balanced_accuracy']:.3f} and "
    f"macro_f1={final_metrics['macro_f1']:.3f}; rationale: {final_reason}."
)
if notes:
    print(" ".join(notes))
print(
    "Use balanced accuracy as the headline metric, macro F1 as the class-balance "
    "check, and the exported confusion matrices plus feature-importance/coefficient "
    "tables for the defence narrative."
)

Final recommendation:
Recommend logistic_regression from Scenario 2 original WAP features. It reaches balanced_accuracy=0.885 and macro_f1=0.870; rationale: within 0.03 balanced_accuracy of the best Scenario 2 model and most explainable.
Logistic Regression is within 0.03 balanced_accuracy of the best original-feature model, so it is the safest explainable choice. PCA with 100 components is within 0.02 balanced_accuracy of the original-feature best, so it is a valid compression strategy. LDA projection with 10 components performs close to the original-feature best, making it a strong supervised dimensionality-reduction option.
Use balanced accuracy as the headline metric, macro F1 as the class-balance check, and the exported confusion matrices plus feature-importance/coefficient tables for the defence narrative.


## Limitations and future work

Results are tied to the official validation split, which is useful because
it captures a real time/device/user shift but still only represents one
campus. Future work should add explicit feature selection, device-aware
validation, coordinate regression, and a small Python Shiny app for model
inspection once the classification story is accepted.